In [ ]:
"""
Qwen3.5-0.8B LoRA Fine-tuning (Unsloth)
RTX 6000 Pro Blackwell (96GB) / chi 환경
completion_only_loss 방식 (prompt-completion 데이터셋)
"""

import shutil, os

shutil.rmtree("/tmp/torchinductor_taeyoung", ignore_errors=True)
shutil.rmtree(os.path.expanduser("~/nfs-mount/chi2027/unsloth_compiled_cache"), ignore_errors=True)

assert not os.path.exists("/tmp/torchinductor_taeyoung"), "inductor 캐시 삭제 실패"
assert not os.path.exists(os.path.expanduser("~/nfs-mount/chi2027/unsloth_compiled_cache")), "unsloth 캐시 삭제 실패"
print("✅ 컴파일 캐시 삭제 확인 완료")

os.environ["UNSLOTH_TARGET_GB"] = "4"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,garbage_collection_threshold:0.6"
os.environ["HF_HOME"] = "/home/taeyoung/nfs-mount/chi2027/.cache/huggingface"

assert os.environ.get("UNSLOTH_TARGET_GB") == "4"
print(f"✅ UNSLOTH_TARGET_GB = {os.environ['UNSLOTH_TARGET_GB']}")

from unsloth import FastModel

import unsloth_zoo.fused_losses.cross_entropy_loss as _ce_loss
_ce_loss.TARGET_GB = "4"
print(f"✅ unsloth TARGET_GB = {_ce_loss.TARGET_GB}")

import torch
from datasets import load_dataset
from unsloth import FastModel
from trl import SFTTrainer, SFTConfig
from transformers import TrainerCallback


class ClearCacheCallback(TrainerCallback):
    def on_step_end(self, args, state, control, **kwargs):
        torch.cuda.empty_cache()


# -------------------------
# Config
# -------------------------
MODEL_ID = "unsloth/Qwen3.5-9B"         # 27B → 9B
DATASET_PATH = "./data/7_qwen_dataset_train.jsonl"
OUTPUT_DIR = "./model/qwen3.5-9b-lora-teloptextgen-ep1"  # 경로도 바꾸기

MAX_SEQ_LENGTH = 131072
#MAX_SEQ_LENGTH = 16384
NUM_EPOCHS = 1
BATCH_SIZE = 1
GRAD_ACCUM = 8
LEARNING_RATE = 7e-5
LORA_RANK = 64
LORA_ALPHA = 128
SAVE_STEPS = 500
LOGGING_STEPS = 10


# -------------------------
# 모델 + 토크나이저
# -------------------------
print("📦 모델 로딩...")
model, tokenizer = FastModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=False,
    load_in_16bit=True,
    full_finetuning=False,
    use_gradient_checkpointing="unsloth",
)

print(model.config._attn_implementation)
# 또는 base_model 경유
print(model.base_model.config._attn_implementation)

# -------------------------
# LoRA
# -------------------------
print("🔧 LoRA 어댑터 부착...")
model = FastModel.get_peft_model(
    model,
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=0.0,
)
model.print_trainable_parameters()


# -------------------------
# 데이터셋 로드 + prompt/completion 변환
# -------------------------
print("📂 데이터셋 로딩...")
dataset = load_dataset("json", data_files=DATASET_PATH, split="train")

_tok = tokenizer.tokenizer if hasattr(tokenizer, "tokenizer") else tokenizer


def to_prompt_completion(example):
    """
    messages를 prompt(system+user)와 completion(assistant)으로 분리.
    prompt는 assistant turn 시작 마커까지 포함하고,
    completion은 순수 assistant 내용만.
    """
    msgs = example["messages"]
    prompt_msgs = [m for m in msgs if m["role"] != "assistant"]
    asst_msgs = [m for m in msgs if m["role"] == "assistant"]
    assert len(asst_msgs) == 1, f"assistant turn이 1개가 아님: {len(asst_msgs)}"

    # prompt: system + user + "<|im_start|>assistant\n" 까지
    prompt_text = _tok.apply_chat_template(
        prompt_msgs,
        tokenize=False,
        add_generation_prompt=True,   # assistant 시작 마커까지 포함
    )

    # completion: assistant 본문 + "<|im_end|>\n"
    # (EOS가 반드시 학습되도록 명시)
    completion_text = asst_msgs[0]["content"] + "<|im_end|>\n"

    return {"prompt": prompt_text, "completion": completion_text}


dataset = dataset.map(
    to_prompt_completion,
    remove_columns=dataset.column_names,
    num_proc=32,
    desc="prompt/completion 변환",
)
print(f"✅ 데이터셋: {len(dataset)}개 샘플")
print(f"   컬럼: {dataset.column_names}")


# -------------------------
# Sanity check: 첫 샘플 확인
# -------------------------
print("\n🔍 첫 샘플 검증")
_s = dataset[0]
print(f"  prompt 끝부분:     {repr(_s['prompt'][-80:])}")
print(f"  completion 앞부분: {repr(_s['completion'][:80])}")
print(f"  completion 끝부분: {repr(_s['completion'][-40:])}")

# prompt가 assistant 마커로 끝나는지
assert _s["prompt"].rstrip().endswith("assistant") or "<|im_start|>assistant" in _s["prompt"][-50:], \
    f"❌ prompt가 assistant 시작 마커로 끝나지 않음: {repr(_s['prompt'][-80:])}"
# completion이 EOS로 끝나는지
assert _s["completion"].rstrip().endswith("<|im_end|>"), \
    f"❌ completion이 EOS로 끝나지 않음: {repr(_s['completion'][-40:])}"
print("  ✅ prompt/completion 경계 OK\n")


# -------------------------
# 토큰 길이 필터링
# -------------------------
print("🔍 토큰 길이 필터링...")


def count_tokens(example):
    p_ids = _tok.encode(example["prompt"], add_special_tokens=False)
    c_ids = _tok.encode(example["completion"], add_special_tokens=False)
    example["n_tokens"] = len(p_ids) + len(c_ids)
    return example


dataset = dataset.map(count_tokens, num_proc=32, desc="토큰 수 계산")
before = len(dataset)
dataset = dataset.filter(lambda x: x["n_tokens"] <= MAX_SEQ_LENGTH, num_proc=32)
dataset = dataset.remove_columns("n_tokens")
print(f"✅ 필터: {before}개 → {len(dataset)}개 ({before - len(dataset)}개 제외)")


# -------------------------
# 학습
# -------------------------
print("🚀 학습 시작...")
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    bf16=True,
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    save_total_limit=50,
    save_only_model=False,
    max_length=MAX_SEQ_LENGTH,
    completion_only_loss=True,   # ← 핵심: completion 부분만 loss
    dataset_num_proc=32,
    dataloader_num_workers=4,
    optim="adamw_8bit",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    report_to="tensorboard",
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset,
    args=training_args,
)

#trainer.train(resume_from_checkpoint=True)
trainer.train()

# -------------------------
# 저장
# -------------------------
print("💾 어댑터 저장...")
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"✅ 완료! 저장 위치: {OUTPUT_DIR}")

✅ 컴파일 캐시 삭제 확인 완료
✅ UNSLOTH_TARGET_GB = 4
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/home/taeyoung/miniconda3/envs/chi/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🦥 Unsloth Zoo will now patch everything to make training faster!
✅ unsloth TARGET_GB = 4
📦 모델 로딩...


In [ ]:
"""
Qwen3.5-9B LoRA 이어서 훈련 (Unsloth)
기존 epoch 1 학습된 어댑터를 로드해서 추가 epoch 학습
"""

import shutil, os

shutil.rmtree("/tmp/torchinductor_taeyoung", ignore_errors=True)
shutil.rmtree(os.path.expanduser("~/nfs-mount/chi2027/unsloth_compiled_cache"), ignore_errors=True)

assert not os.path.exists("/tmp/torchinductor_taeyoung"), "inductor 캐시 삭제 실패"
assert not os.path.exists(os.path.expanduser("~/nfs-mount/chi2027/unsloth_compiled_cache")), "unsloth 캐시 삭제 실패"
print("✅ 컴파일 캐시 삭제 확인 완료")

os.environ["UNSLOTH_TARGET_GB"] = "4"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,garbage_collection_threshold:0.6"
os.environ["HF_HOME"] = "/home/taeyoung/nfs-mount/chi2027/.cache/huggingface"

assert os.environ.get("UNSLOTH_TARGET_GB") == "4"
print(f"✅ UNSLOTH_TARGET_GB = {os.environ['UNSLOTH_TARGET_GB']}")

from unsloth import FastModel

import unsloth_zoo.fused_losses.cross_entropy_loss as _ce_loss
_ce_loss.TARGET_GB = "4"
print(f"✅ unsloth TARGET_GB = {_ce_loss.TARGET_GB}")

import torch
from datasets import load_dataset
from unsloth import FastModel
from trl import SFTTrainer, SFTConfig
from transformers import TrainerCallback


class ClearCacheCallback(TrainerCallback):
    def on_step_end(self, args, state, control, **kwargs):
        torch.cuda.empty_cache()


# -------------------------
# Config
# -------------------------
PREV_LORA_DIR = "./model/qwen3.5-9b-lora-teloptextgen-ep1"        # 기존 학습 결과
OUTPUT_DIR    = "./model/qwen3.5-9b-lora-teloptextgen-ep2"    # 새 저장 경로
DATASET_PATH  = "./data/7_qwen_dataset_train.jsonl"

MAX_SEQ_LENGTH = 16384
NUM_EPOCHS = 1          # 추가 1 epoch (누적 2 epoch 효과)
BATCH_SIZE = 1
GRAD_ACCUM = 8
LEARNING_RATE = 3e-5    # 기존 7e-5의 약 절반, 미세 조정용
LORA_RANK = 64
LORA_ALPHA = 128
SAVE_STEPS = 500
LOGGING_STEPS = 10


# -------------------------
# 기존 LoRA 어댑터를 붙인 모델 로드
# -------------------------
print(f"📦 기존 LoRA 어댑터 로딩: {PREV_LORA_DIR}")
model, tokenizer = FastModel.from_pretrained(
    model_name=PREV_LORA_DIR,       # ← base가 아니라 기존 어댑터 경로
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=False,
    load_in_16bit=True,
    full_finetuning=False,
    use_gradient_checkpointing="unsloth",
)

print(model.config._attn_implementation)
print(model.base_model.config._attn_implementation)

# 어댑터 이미 로드됨 → get_peft_model 호출하지 않음
# 그래디언트가 LoRA 파라미터에만 흐르는지 확인
model.print_trainable_parameters()


# -------------------------
# 데이터셋 로드 + prompt/completion 변환
# -------------------------
print("📂 데이터셋 로딩...")
dataset = load_dataset("json", data_files=DATASET_PATH, split="train")

_tok = tokenizer.tokenizer if hasattr(tokenizer, "tokenizer") else tokenizer


def to_prompt_completion(example):
    msgs = example["messages"]
    prompt_msgs = [m for m in msgs if m["role"] != "assistant"]
    asst_msgs = [m for m in msgs if m["role"] == "assistant"]
    assert len(asst_msgs) == 1, f"assistant turn이 1개가 아님: {len(asst_msgs)}"

    prompt_text = _tok.apply_chat_template(
        prompt_msgs,
        tokenize=False,
        add_generation_prompt=True,
    )
    completion_text = asst_msgs[0]["content"] + "<|im_end|>\n"
    return {"prompt": prompt_text, "completion": completion_text}


dataset = dataset.map(
    to_prompt_completion,
    remove_columns=dataset.column_names,
    num_proc=32,
    desc="prompt/completion 변환",
)
print(f"✅ 데이터셋: {len(dataset)}개 샘플")
print(f"   컬럼: {dataset.column_names}")


# -------------------------
# Sanity check
# -------------------------
print("\n🔍 첫 샘플 검증")
_s = dataset[0]
print(f"  prompt 끝부분:     {repr(_s['prompt'][-80:])}")
print(f"  completion 앞부분: {repr(_s['completion'][:80])}")
print(f"  completion 끝부분: {repr(_s['completion'][-40:])}")

assert _s["prompt"].rstrip().endswith("assistant") or "<|im_start|>assistant" in _s["prompt"][-50:], \
    f"❌ prompt가 assistant 시작 마커로 끝나지 않음: {repr(_s['prompt'][-80:])}"
assert _s["completion"].rstrip().endswith("<|im_end|>"), \
    f"❌ completion이 EOS로 끝나지 않음: {repr(_s['completion'][-40:])}"
print("  ✅ prompt/completion 경계 OK\n")


# -------------------------
# 토큰 길이 필터링
# -------------------------
print("🔍 토큰 길이 필터링...")


def count_tokens(example):
    p_ids = _tok.encode(example["prompt"], add_special_tokens=False)
    c_ids = _tok.encode(example["completion"], add_special_tokens=False)
    example["n_tokens"] = len(p_ids) + len(c_ids)
    return example


dataset = dataset.map(count_tokens, num_proc=32, desc="토큰 수 계산")
before = len(dataset)
dataset = dataset.filter(lambda x: x["n_tokens"] <= MAX_SEQ_LENGTH, num_proc=32)
dataset = dataset.remove_columns("n_tokens")
print(f"✅ 필터: {before}개 → {len(dataset)}개 ({before - len(dataset)}개 제외)")


# -------------------------
# 학습
# -------------------------
print("🚀 이어서 학습 시작...")
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.01,              # 0.05 → 0.01 (이미 데워진 상태)
    bf16=True,
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    save_total_limit=50,
    save_only_model=False,
    max_length=MAX_SEQ_LENGTH,
    completion_only_loss=True,
    dataset_num_proc=32,
    dataloader_num_workers=4,
    optim="adamw_8bit",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    report_to="tensorboard",
    seed=43,                        # 기존과 다른 seed → 다른 순서로 데이터 노출
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset,
    args=training_args,
)

trainer.train()


# -------------------------
# 저장
# -------------------------
print("💾 어댑터 저장...")
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"✅ 완료! 저장 위치: {OUTPUT_DIR}")

In [ ]:
"""
Qwen3.5-9B LoRA 이어서 훈련 - Epoch 3 (ep2 → ep3)
"""

import shutil, os

shutil.rmtree("/tmp/torchinductor_taeyoung", ignore_errors=True)
shutil.rmtree(os.path.expanduser("~/nfs-mount/chi2027/unsloth_compiled_cache"), ignore_errors=True)

assert not os.path.exists("/tmp/torchinductor_taeyoung"), "inductor 캐시 삭제 실패"
assert not os.path.exists(os.path.expanduser("~/nfs-mount/chi2027/unsloth_compiled_cache")), "unsloth 캐시 삭제 실패"
print("✅ 컴파일 캐시 삭제 확인 완료")

os.environ["UNSLOTH_TARGET_GB"] = "4"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,garbage_collection_threshold:0.6"
os.environ["HF_HOME"] = "/home/taeyoung/nfs-mount/chi2027/.cache/huggingface"

assert os.environ.get("UNSLOTH_TARGET_GB") == "4"
print(f"✅ UNSLOTH_TARGET_GB = {os.environ['UNSLOTH_TARGET_GB']}")

from unsloth import FastModel

import unsloth_zoo.fused_losses.cross_entropy_loss as _ce_loss
_ce_loss.TARGET_GB = "4"
print(f"✅ unsloth TARGET_GB = {_ce_loss.TARGET_GB}")

import torch
from datasets import load_dataset
from unsloth import FastModel
from trl import SFTTrainer, SFTConfig
from transformers import TrainerCallback


class ClearCacheCallback(TrainerCallback):
    def on_step_end(self, args, state, control, **kwargs):
        torch.cuda.empty_cache()


# -------------------------
# Config
# -------------------------
PREV_LORA_DIR = "./model/qwen3.5-9b-lora-teloptextgen-ep2"     # ep2 결과
OUTPUT_DIR    = "./model/qwen3.5-9b-lora-teloptextgen-ep3"     # 새 저장 경로
DATASET_PATH  = "./data/7_qwen_dataset_train.jsonl"

MAX_SEQ_LENGTH = 16384

NUM_EPOCHS = 1          # 추가 1 epoch (누적 3 epoch 효과)
BATCH_SIZE = 1
GRAD_ACCUM = 8
LEARNING_RATE = 1.5e-5  # ep2의 3e-5 → 절반. 수렴 후반에 더 작게
LORA_RANK = 64
LORA_ALPHA = 128
SAVE_STEPS = 500
LOGGING_STEPS = 10


# -------------------------
# 기존 LoRA 어댑터 로드
# -------------------------
print(f"📦 기존 LoRA 어댑터 로딩: {PREV_LORA_DIR}")
model, tokenizer = FastModel.from_pretrained(
    model_name=PREV_LORA_DIR,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=False,
    load_in_16bit=True,
    full_finetuning=False,
    use_gradient_checkpointing="unsloth",
)

print(model.config._attn_implementation)
print(model.base_model.config._attn_implementation)

model.print_trainable_parameters()


# -------------------------
# 데이터셋 로드 + prompt/completion 변환
# -------------------------
print("📂 데이터셋 로딩...")
dataset = load_dataset("json", data_files=DATASET_PATH, split="train")

_tok = tokenizer.tokenizer if hasattr(tokenizer, "tokenizer") else tokenizer


def to_prompt_completion(example):
    msgs = example["messages"]
    prompt_msgs = [m for m in msgs if m["role"] != "assistant"]
    asst_msgs = [m for m in msgs if m["role"] == "assistant"]
    assert len(asst_msgs) == 1, f"assistant turn이 1개가 아님: {len(asst_msgs)}"

    prompt_text = _tok.apply_chat_template(
        prompt_msgs,
        tokenize=False,
        add_generation_prompt=True,
    )
    completion_text = asst_msgs[0]["content"] + "<|im_end|>\n"
    return {"prompt": prompt_text, "completion": completion_text}


dataset = dataset.map(
    to_prompt_completion,
    remove_columns=dataset.column_names,
    num_proc=32,
    desc="prompt/completion 변환",
)

# ep1(seed 42), ep2(seed 43)와 다른 순서로 섞기
dataset = dataset.shuffle(seed=44)

print(f"✅ 데이터셋: {len(dataset)}개 샘플")
print(f"   컬럼: {dataset.column_names}")


# -------------------------
# Sanity check
# -------------------------
print("\n🔍 첫 샘플 검증")
_s = dataset[0]
print(f"  prompt 끝부분:     {repr(_s['prompt'][-80:])}")
print(f"  completion 앞부분: {repr(_s['completion'][:80])}")
print(f"  completion 끝부분: {repr(_s['completion'][-40:])}")

assert _s["prompt"].rstrip().endswith("assistant") or "<|im_start|>assistant" in _s["prompt"][-50:], \
    f"❌ prompt가 assistant 시작 마커로 끝나지 않음: {repr(_s['prompt'][-80:])}"
assert _s["completion"].rstrip().endswith("<|im_end|>"), \
    f"❌ completion이 EOS로 끝나지 않음: {repr(_s['completion'][-40:])}"
print("  ✅ prompt/completion 경계 OK\n")


# -------------------------
# 토큰 길이 필터링
# -------------------------
print("🔍 토큰 길이 필터링...")


def count_tokens(example):
    p_ids = _tok.encode(example["prompt"], add_special_tokens=False)
    c_ids = _tok.encode(example["completion"], add_special_tokens=False)
    example["n_tokens"] = len(p_ids) + len(c_ids)
    return example


dataset = dataset.map(count_tokens, num_proc=32, desc="토큰 수 계산")
before = len(dataset)
dataset = dataset.filter(lambda x: x["n_tokens"] <= MAX_SEQ_LENGTH, num_proc=32)
dataset = dataset.remove_columns("n_tokens")
print(f"✅ 필터: {before}개 → {len(dataset)}개 ({before - len(dataset)}개 제외)")


# -------------------------
# 학습
# -------------------------
print("🚀 ep3 학습 시작...")
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.01,
    bf16=True,
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    save_total_limit=50,
    save_only_model=False,
    max_length=MAX_SEQ_LENGTH,
    completion_only_loss=True,
    dataset_num_proc=32,
    dataloader_num_workers=4,
    optim="adamw_8bit",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    report_to="tensorboard",
    seed=44,
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset,
    args=training_args,
)

trainer.train()


# -------------------------
# 저장
# -------------------------
print("💾 어댑터 저장...")
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"✅ 완료! 저장 위치: {OUTPUT_DIR}")

In [ ]:
# %% LoRA merge
from unsloth import FastModel
from transformers import AutoTokenizer
import os

BASE_MODEL = "Qwen/Qwen3.5-9B"
LORA_DIR = "./model/qwen3.5-9b-lora-teloptextgen-rank64-split-16K-datacleaning"
MERGED_DIR = "./model/qwen3.5-9b-lora-teloptextgen-rank64-split-16K-datacleaning-merged"

model, tokenizer = FastModel.from_pretrained(
    model_name=LORA_DIR,
    max_seq_length=16384,
    load_in_4bit=False,
    load_in_16bit=True,
    full_finetuning=False,
)

print("📦 Merge 시작...")
model.save_pretrained_merged(
    MERGED_DIR,
    tokenizer,
    save_method="merged_16bit",
)

print("🔧 토크나이저를 표준 HF 포맷으로 재저장...")
base_tok = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=False)
for fn in [
    "tokenizer.json", "tokenizer_config.json",
    "special_tokens_map.json", "vocab.json", "merges.txt",
    "added_tokens.json", "chat_template.jinja",
]:
    p = os.path.join(MERGED_DIR, fn)
    if os.path.exists(p):
        os.remove(p)
base_tok.save_pretrained(MERGED_DIR)

print(f"✅ 완료: {MERGED_DIR}")
print(f"   파일 목록: {sorted(os.listdir(MERGED_DIR))}")

In [ ]:
# %% LoRA merge
from unsloth import FastModel
from transformers import AutoTokenizer
import os

BASE_MODEL = "Qwen/Qwen3.5-9B"
LORA_DIR = "./model/qwen3.5-9b-lora-teloptextgen-rank64-split-16K-datacleaning-ep2"
MERGED_DIR = "./model/qwen3.5-9b-lora-teloptextgen-rank64-split-16K-datacleaning-ep2-merged"

model, tokenizer = FastModel.from_pretrained(
    model_name=LORA_DIR,
    max_seq_length=16384,
    load_in_4bit=False,
    load_in_16bit=True,
    full_finetuning=False,
)

print("📦 Merge 시작...")
model.save_pretrained_merged(
    MERGED_DIR,
    tokenizer,
    save_method="merged_16bit",
)

print("🔧 토크나이저를 표준 HF 포맷으로 재저장...")
base_tok = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=False)
for fn in [
    "tokenizer.json", "tokenizer_config.json",
    "special_tokens_map.json", "vocab.json", "merges.txt",
    "added_tokens.json", "chat_template.jinja",
]:
    p = os.path.join(MERGED_DIR, fn)
    if os.path.exists(p):
        os.remove(p)
base_tok.save_pretrained(MERGED_DIR)

print(f"✅ 완료: {MERGED_DIR}")
print(f"   파일 목록: {sorted(os.listdir(MERGED_DIR))}")

In [ ]:
# %% LoRA merge
from unsloth import FastModel
from transformers import AutoTokenizer
import os

BASE_MODEL = "Qwen/Qwen3.5-9B"
LORA_DIR = "./model/qwen3.5-9b-lora-teloptextgen-rank64-split-16K-datacleaning-ep3"
MERGED_DIR = "./model/qwen3.5-9b-lora-teloptextgen-rank64-split-16K-datacleaning-ep3-merged"

model, tokenizer = FastModel.from_pretrained(
    model_name=LORA_DIR,
    max_seq_length=16384,
    load_in_4bit=False,
    load_in_16bit=True,
    full_finetuning=False,
)

print("📦 Merge 시작...")
model.save_pretrained_merged(
    MERGED_DIR,
    tokenizer,
    save_method="merged_16bit",
)

print("🔧 토크나이저를 표준 HF 포맷으로 재저장...")
base_tok = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=False)
for fn in [
    "tokenizer.json", "tokenizer_config.json",
    "special_tokens_map.json", "vocab.json", "merges.txt",
    "added_tokens.json", "chat_template.jinja",
]:
    p = os.path.join(MERGED_DIR, fn)
    if os.path.exists(p):
        os.remove(p)
base_tok.save_pretrained(MERGED_DIR)

print(f"✅ 완료: {MERGED_DIR}")
print(f"   파일 목록: {sorted(os.listdir(MERGED_DIR))}")

In [ ]:
import json

MERGED_DIR = "./model/qwen3.5-9b-lora-teloptextgen-rank64-split-16K-datacleaning-merged"

with open(f"{MERGED_DIR}/tokenizer_config.json", "r") as f:
    config = json.load(f)

print(f"현재: {config.get('tokenizer_class')}")
config["tokenizer_class"] = "Qwen2Tokenizer"

with open(f"{MERGED_DIR}/tokenizer_config.json", "w") as f:
    json.dump(config, f, indent=2, ensure_ascii=False)

print("✅ 수정 완료")

In [ ]:
import json

MERGED_DIR = "./model/qwen3.5-9b-lora-teloptextgen-rank64-split-16K-datacleaning-ep2-merged"

with open(f"{MERGED_DIR}/tokenizer_config.json", "r") as f:
    config = json.load(f)

print(f"현재: {config.get('tokenizer_class')}")
config["tokenizer_class"] = "Qwen2Tokenizer"

with open(f"{MERGED_DIR}/tokenizer_config.json", "w") as f:
    json.dump(config, f, indent=2, ensure_ascii=False)

print("✅ 수정 완료")

In [ ]:
import json

MERGED_DIR = "./model/qwen3.5-9b-lora-teloptextgen-rank64-split-16K-datacleaning-ep3-merged"

with open(f"{MERGED_DIR}/tokenizer_config.json", "r") as f:
    config = json.load(f)

print(f"현재: {config.get('tokenizer_class')}")
config["tokenizer_class"] = "Qwen2Tokenizer"

with open(f"{MERGED_DIR}/tokenizer_config.json", "w") as f:
    json.dump(config, f, indent=2, ensure_ascii=False)

print("✅ 수정 완료")